<a href="https://colab.research.google.com/github/beatriz-paz/learning-cnn/blob/main/cnn_customizing_layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Criando uma camada Wavelet usando TensorFlow

In [ ]:
# Camada Wavelet criada
import tensorflow as tf

class WaveletLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(WaveletLayer, self).__init__()

        # Filtro Haar fixo
        self.kernel = tf.constant(
            [[1/2, 1/2],
             [1/2, 1/2]],
            dtype=tf.float32
        )
        self.kernel = tf.reshape(self.kernel, [2, 2, 1, 1])

    def call(self, inputs):
        return tf.nn.conv2d(inputs, self.kernel, strides=2, padding='SAME')


## Exemplo de como usar a camada Wavelet no modelo CNN:

In [ ]:
# Agora a Wavelet faz parte do pipeline
model = tf.keras.Sequential([
    WaveletLayer(),
    tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(10, activation='softmax')
])


# Criando uma camada Wavelet usando PyTorch

In [ ]:
# Camada Wavelet criada
import torch
import torch.nn as nn
import torch.nn.functional as F

class WaveletLayer(nn.Module):
    def __init__(self):
        super(WaveletLayer, self).__init__()

        # Filtro Haar LL
        kernel = torch.tensor([[0.5, 0.5],
                               [0.5, 0.5]], dtype=torch.float32)

        # Ajustar formato: (out_channels, in_channels, H, W)
        kernel = kernel.view(1, 1, 2, 2)

        # Registrar como peso fixo (não treinável)
        self.register_buffer("weight", kernel)
        # para tornar a wavelet treinável pela própria cnn: self.weight = nn.Parameter(kernel)

    def forward(self, x):
        return F.conv2d(x, self.weight, stride=2)


## Exemplo de como usar a camada Wavelet no modelo CNN:

In [ ]:
class WaveletCNN(nn.Module):
    def __init__(self):
        super(WaveletCNN, self).__init__()

        self.wavelet = WaveletLayer()
        self.conv = nn.Conv2d(1, 16, 3, padding=1)
        self.fc = nn.Linear(16 * 14 * 14, 10)

    def forward(self, x):
        x = self.wavelet(x)  # Wavelet primeiro
        x = torch.relu(self.conv(x))
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


### Referências:

TensorFlow:

* https://www.youtube.com/watch?v=uSklwAAA1Zg
* https://www.piyush-yadav.com/p/creating-custom-layers-in-keras-a

PyTorch:

* https://discuss.pytorch.org/t/how-to-implement-a-custom-convolutional-layer-and-call-it-from-your-own-network/120196